<a href="https://colab.research.google.com/github/KhadijaZaib/Bahria-University-Management-System/blob/main/02kinshipFace_Detection_and_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# CELL 1: Load the previously prepared kinship dataset.
#
# This code:
# 1. Connects Google Colab to Google Drive.
# 2. Loads our corrected Kin/Non-Kin candidate pairs.
# 3. Loads the saved image metadata.
# 4. Restores the selected image pairs.
# 5. Prepares 12 sample pairs for face detection.
#
# IMPORTANT:
# No new CSV files are created.
# No previous data is overwritten.
# Family-wise train/validation/test splits remain unchanged.

import pandas as pd
import numpy as np
from google.colab import drive

# Connect Google Drive
drive.mount("/content/drive")

# Existing project location
project = (
    "/content/drive/MyDrive/"
    "Kinship Verification under Cosmetic Alteration FYP"
)

# Load corrected binary candidate dataset
df_pairs = pd.read_csv(
    project +
    "/checkpoints/"
    "binary_dataset_20261123_004139_820411/"
    "corrected_binary_kinship_candidates.csv"
)

# Load previously saved image metadata
df_metadata = pd.read_csv(
    project +
    "/checkpoints/"
    "backup_20261123_002815/"
    "image_metadata.csv"
)

# Build image lists for each unique person
person_images = (
    df_metadata.dropna(subset=["image_path"])
    .groupby(["family", "person"])["image_path"]
    .apply(lambda x: sorted(set(x)))
    .to_dict()
)

# Attach image lists for both persons
df_images = df_pairs.copy()

df_images["person1_images"] = [
    person_images.get((f, p), [])
    for f, p in zip(
        df_images["family1"],
        df_images["person1"]
    )
]

df_images["person2_images"] = [
    person_images.get((f, p), [])
    for f, p in zip(
        df_images["family2"],
        df_images["person2"]
    )
]

# Select one reproducible image for each person
rng = np.random.default_rng(42)

def select_image(images):
    if not images:
        return None
    return images[rng.integers(len(images))]

df_sample57 = df_images.copy()

df_sample57["image1_path"] = (
    df_sample57["person1_images"].apply(select_image)
)

df_sample57["image2_path"] = (
    df_sample57["person2_images"].apply(select_image)
)

# Recreate the 12-pair face detection sample
df_test58 = (
    df_sample57.groupby("split", group_keys=False)
    .sample(n=4, random_state=42)
    .copy()
)

# Check saved dataset and image selections
assert len(df_pairs) == 1691
assert df_sample57["image1_path"].notna().all()
assert df_sample57["image2_path"].notna().all()

# Check that family assignments remain separate
family_splits = pd.concat([
    df_pairs[["family1", "split"]].rename(
        columns={"family1": "family"}
    ),
    df_pairs[["family2", "split"]].rename(
        columns={"family2": "family"}
    )
])

assert (
    family_splits.groupby("family")["split"]
    .nunique().max() == 1
)

print("PROJECT RESTORED SUCCESSFULLY!")
print("Total candidate pairs:", len(df_pairs))
print("Kin pairs:", (df_pairs["label"] == 1).sum())
print("Non-Kin pairs:", (df_pairs["label"] == 0).sum())
print("Face detection sample pairs:", len(df_test58))
print("Family overlap: 0")
print("No CSV files created or overwritten.")
print("\nREADY FOR FACE DETECTION!")


Mounted at /content/drive
PROJECT RESTORED SUCCESSFULLY!
Total candidate pairs: 1691
Kin pairs: 845
Non-Kin pairs: 846
Face detection sample pairs: 12
Family overlap: 0
No CSV files created or overwritten.

READY FOR FACE DETECTION!
